In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Math
import pickle
import os

In [ ]:

# ==========================
# Setup and Data Loading
# ==========================

# --- Results Directory and Output Setup ---
results_dir = "../../Results/Data/Complete_rho/"

Output_dir = "../../Results/Plot/Populations"
os.makedirs(Output_dir, exist_ok=True)

# --- Time Step and Trajectory String Formatting ---

dt = 0.01
N_traj = 20000
dt_str = f"{dt:.6f}".replace(".", "p")

# Load Quantum Jump (QJ) Data
fname_QJ = os.path.join(results_dir, f"result_mode_QJ_dt{dt_str}_Ntraj{N_traj}.npz")
data_QJ = np.load(fname_QJ)

# Load State Diffusion (SD) Data
fname_SD = os.path.join(results_dir, f"result_mode_SD_dt{dt_str}_Ntraj{N_traj}.npz")
data_SD = np.load(fname_SD)

print("Data extraction completed")



In [ ]:
# ================================
# Data Extraction and Preparation
# ================================

times = data_QJ['times']

# 11 -> excited sstate, 00 -> ground state, 01 -> coherence

# Extract averages for QJ over all trajectories
avg_pop_00_QJ = np.mean(data_QJ['pop_00'], axis=1)
avg_pop_11_QJ = np.mean(data_QJ['pop_11'], axis=1)
avg_coh_01_QJ = np.mean(data_QJ['coh_01'], axis=1)

# Extract samples for QJ over all trajectories
sample_traj_pop_00_QJ = data_QJ['pop_00'][:, :50]  # First 50 trajectories
sample_traj_pop_11_QJ = data_QJ['pop_11'][:, :50]  # First 50 trajectories
sample_traj_coh_01_QJ = data_QJ['coh_01'][:, :50]  # First 50 trajectories

# Extract averages for SD over all trajectories
avg_pop_00_SD = np.mean(data_SD['pop_00'], axis=1)
avg_pop_11_SD = np.mean(data_SD['pop_11'], axis=1)
avg_coh_01_SD = np.mean(data_SD['coh_01'], axis=1)

# Extract samples for SD over all trajectories
sample_traj_pop_00_SD = data_SD['pop_00'][:, :50]  # First 50 trajectories
sample_traj_pop_11_SD = data_SD['pop_11'][:, :50]  # First 50 trajectories
sample_traj_coh_01_SD = data_SD['coh_01'][:, :50]  # First 50 trajectories

# Extract analytical baselines (same for both files, we can just read from QJ)
pops_trace_00 = data_QJ['pops_trace'][0, :]
pops_trace_11 = data_QJ['pops_trace'][1, :]
cohe_trace_01 = data_QJ['cohe_trace'][0, :]
pop_traj_isolated_00 = data_QJ['pop_traj_isolated'][0, :]
pop_traj_isolated_11 = data_QJ['pop_traj_isolated'][1, :]

# Extract Lindblad populations
rho_list_lindblad = data_QJ['rho_list_lindblad']
lindblad_00 = np.real(rho_list_lindblad[:, 0, 0])
lindblad_11 = np.real(rho_list_lindblad[:, 1, 1])
lindblad_01 = rho_list_lindblad[:, 0, 1]

In [ ]:
# ===========================
# General Setup for Plotting
# ===========================

# --- Global Style Settings (Matplotlib rcParams) ---
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.figsize': (8, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': ':',
    'figure.autolayout': True # plt.tight_layout()
})

# --- Automatic Figure Saving Function ---
def save_fig(fig, filename):
    """
    Saves the figure in both PNG or PDF
    """
    path_png = os.path.join(Output_dir, f"{filename}.png")
    # path_pdf = os.path.join(Output_dir, f"{filename}.pdf")  # save in pdf
    
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    # fig.savefig(path_pdf, bbox_inches='tight') # save in pdf
    print(f"Figure saved in: {Output_dir}/{filename}")

In [ ]:
# ====================================
# Convergence Avg vs Trace vs Linblad
# ====================================

plt.close('all')

fig01, ax = plt.subplots()

ax.plot(times, lindblad_11, label=r'Lindblad', linewidth=2, linestyle='--')
ax.plot(times, pops_trace_11, label=r'Anc_trace', linewidth=2, linestyle=':')
ax.plot(times, avg_pop_11_SD, label=r'Avg_traj', linewidth=2, alpha=0.5)

ax.set_title(f'Comparison Lindblad, Trace, Avg Trajectories | dt={dt} & N_traj={N_traj}')
ax.set_xlabel('Time')
ax.set_ylabel(f'Population state |1> ') # change site
ax.legend()

# Automatically save the figure (PNG + PDF)
filename_01 = f'SD Comparison Lindblad, Trace, Avg Trajectories - dt={dt} & N_traj={N_traj}'
save_fig(fig01, filename_01)

plt.show()

In [ ]:
# ================================================
# Comparison trajectories Collisional vs Isolated
# ================================================

plt.close('all')

sample_idx = 1

fig02, ax = plt.subplots(figsize=(10,5))

ax.plot(times, sample_traj_pop_11_SD[:,sample_idx], label='Single Traj', linewidth=2)
ax.plot(times, pop_traj_isolated_11, label='Traj Isolated', linewidth=2, linestyle=':')

ax.set_title(r'Comparison trajectories Collisional vs Isolated')
ax.set_xlabel('Time')
ax.set_ylabel(f'Population site 1 ')
ax.legend()

# Automatically save the figure (PNG + PDF)
filename_02 = f'SD Comparison Collisional vs Isolated - dt={dt} & N_traj={N_traj}'
#save_fig(fig02, filename_02)

plt.show()


In [ ]:
# ===================================================
# Comparison Single Trajectory vs Average vs Lindblad
# ===================================================

plt.close('all')

fig03, ax = plt.subplots(figsize=(10, 5))

for i in range(sample_traj_pop_11_SD.shape[1]):
    ax.plot(times, sample_traj_pop_11_SD[:, i], color='gray', alpha=0.15, linewidth=0.5, 
             label='Single Traj' if i==0 else "")

ax.plot(times, lindblad_11, label=r'Lindblad', linewidth=2, linestyle='--')
ax.plot(times, avg_pop_11_SD, label=r'Avg_traj', linewidth=2, color='orange',  alpha=0.5)

ax.set_title(f'Comparison Many Trajectories vs Average - dt={dt} & N_traj={N_traj}')
ax.set_xlabel('Time')
ax.set_ylabel(f'Population site 1 ') # change site
ax.legend()

# Automatically save the figure (PNG + PDF)
filename_03 = 'SD Comparison Many Trajectories vs Average'
save_fig(fig03, filename_03)
plt.show()

In [ ]:
plt.close('all')

fig04, ax = plt.subplots(figsize=(10, 5))

# -----------
# Populations
# -----------
#ax.plot(times, avg_pop_11_SD, label=r'pop 1')

#ax.plot(times, lindblad_11, label=r'pop 1 Lindblad', linewidth=2, linestyle='--')

# ---------------
# Real Coherences
# ---------------
#ax.plot(times, np.real(avg_coh_01_SD), label=r'cohe 1')

#ax.plot(times, np.real(lindblad_01), label=r'cohe 1 Lindblad', linewidth=2, linestyle='--')

# --------------------
# Imaginary Coherences
# --------------------
ax.plot(times, np.imag(avg_coh_01_SD), label=r'cohe 1')

ax.plot(times, np.imag(lindblad_01), label=r'cohe 1 Lindblad', linewidth=2, linestyle='--')

ax.set_title(f'Comparison Lindblad vs Average, Pop, Real and Imag Cohe - dt={dt} & N_traj={N_traj} ')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend()

plt.show()